# 家計調査データ概要・傾向分析(Google Colab版)

**事前準備(このノートブックを開く前に1回だけ):**

1. ローカルの `data/` フォルダ(以下の5ファイルが入っている)を、Googleドライブのマイドライブ直下に**フォルダ名`data`のまま**アップロード
   - `kakei_savings_debt_by_income.csv`(貯蓄・負債: 年収階級別)
   - `kakei_savings_debt_by_age.csv`(貯蓄・負債: 年齢階級別)
   - `kakei_savings_breakdown_by_age.csv`(貯蓄の内訳: 年齢階級別)
   - `kakei_income_expense_by_age.csv`(月々の収支: 年齢5歳刻み)
   - `kakei_surplus_rate_by_income_quintile.csv`(月々の収支: 年収五分位別)
2. 下のセルを上から順に実行(最初のセルでDriveへのアクセス許可を求められるので許可)

> マイドライブの別の場所に置いた場合は、下の `CANDIDATE_DIRS` に自分のパスを追加してください。

In [ ]:
import os

IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# 上から順に探して最初に見つかったフォルダを使う
CANDIDATE_DIRS = [
    "data",                                  # ノートブックと同じ階層に data/ がある場合(ローカル実行時)
    "../data",                                # notebooks/ から起動した場合
    "/content/drive/MyDrive/data",            # Driveのマイドライブ直下に data フォルダを置いた場合
    "/content/data",                          # files.upload() でColabセッションに直接置いた場合
]

DATA_DIR = None
for d in CANDIDATE_DIRS:
    if os.path.isdir(d) and os.path.exists(os.path.join(d, "kakei_savings_debt_by_income.csv")):
        DATA_DIR = d
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "data フォルダが見つかりません。CANDIDATE_DIRS に自分の実際のパスを追加するか、\n"
        "Googleドライブのマイドライブ直下に 'data' という名前でフォルダをアップロードしてください。"
    )

print("使用するデータフォルダ:", DATA_DIR)

## 1. データ読み込み

In [ ]:
import pandas as pd

df_savings = pd.read_csv(f"{DATA_DIR}/kakei_savings_debt_by_income.csv")        # 貯蓄・負債(年収階級別)
df_savings_age = pd.read_csv(f"{DATA_DIR}/kakei_savings_debt_by_age.csv")       # 貯蓄・負債(年齢階級別)
df_breakdown = pd.read_csv(f"{DATA_DIR}/kakei_savings_breakdown_by_age.csv")    # 貯蓄の内訳(年齢階級別)
df_flow_age = pd.read_csv(f"{DATA_DIR}/kakei_income_expense_by_age.csv")        # 月々の収支(年齢5歳刻み)
df_surplus = pd.read_csv(f"{DATA_DIR}/kakei_surplus_rate_by_income_quintile.csv")  # 月々の収支(年収五分位別)

for name, df in [("貯蓄・負債(年収別)", df_savings), ("貯蓄・負債(年齢別)", df_savings_age),
                 ("貯蓄の内訳(年齢別)", df_breakdown), ("月々の収支(年齢別)", df_flow_age),
                 ("月々の収支(年収五分位別)", df_surplus)]:
    print(f"{name}: {df.shape}")

## 2. データセットの概要

**出典: 総務省統計局「家計調査」(e-Stat)**

1つの調査(毎月約9,000世帯の家計簿)を、総務省が軸ごとに集計・公表した統計表から取得している。
**1統計表=1ファイル**で持つ(平均値のみの公表なので、年齢×年収などのクロス集計はできない)。

| ファイル | 統計表ID | 内容 | 対象世帯 | 期間 |
|---|---|---|---|---|
| kakei_savings_debt_by_income.csv | 0002210009(貯蓄・負債編) | 貯蓄・負債の残高(年収18階級) | 二人以上の世帯 | 2002〜2025年(四半期) |
| kakei_savings_debt_by_age.csv | 0002210017(貯蓄・負債編) | 貯蓄・負債の残高(年齢6階級) | 二人以上の世帯 | 2002〜2025年(四半期) |
| kakei_savings_breakdown_by_age.csv | 0002210017(貯蓄・負債編) | 貯蓄の内訳5種(年齢6階級) | 二人以上の世帯 | 2002〜2025年(四半期) |
| kakei_income_expense_by_age.csv | 0002070010(家計収支編) | 月々の収支6項目(年齢5歳刻み) | 勤労者世帯 | 2000〜2026年(月次) |
| kakei_surplus_rate_by_income_quintile.csv | 0002200003(家計収支編) | 月々の収支6項目(年収五分位) | 勤労者世帯 | 2000〜2026年(四半期) |

- **残高(ストック)**は「万円」単位、**月々の収支(フロー)**は「円」単位(黒字率・平均消費性向は%)
- 月々の収支は勤労者世帯のみ(黒字率が定義できるのは給与収入がある世帯のため)。70歳以上の勤労者世帯はサンプルが少なく未公表

In [ ]:
def describe_columns(df: pd.DataFrame, name: str):
    print(f"=== {name} ===")
    print("列:", list(df.columns))
    print("行数:", len(df))
    for col in df.columns:
        if df[col].dtype == object or df[col].nunique() < 20:
            uniques = df[col].unique()
            print(f"  [{col}] ユニーク数={df[col].nunique()} -> {list(uniques)[:20]}")
        else:
            print(f"  [{col}] dtype={df[col].dtype}, min={df[col].min()}, max={df[col].max()}")
    print()

describe_columns(df_savings, "貯蓄・負債(年収別)")
describe_columns(df_savings_age, "貯蓄・負債(年齢別)")
describe_columns(df_breakdown, "貯蓄の内訳(年齢別)")
describe_columns(df_flow_age, "月々の収支(年齢別)")
describe_columns(df_surplus, "月々の収支(年収五分位別)")

## 2.5 日本語フォント設定(文字化け対策)

Colab(および多くのLinux環境)のmatplotlibはデフォルトで日本語グリフを含むフォントを持っていないため、
グラフのタイトルや軸ラベルが豆腐(□□□)になる。`matplotlib-fontja` を読み込むだけで解消する。

> 旧来の `japanize-matplotlib` は Python 3.12以降で動かない(`distutils` 依存)ため、後継の `matplotlib-fontja` を使用。

In [ ]:
try:
    import matplotlib_fontja
except ImportError:
    %pip install -q matplotlib-fontja
    import matplotlib_fontja

## 3. 年収階級別 貯蓄・負債(全期間平均)

In [ ]:
INCOME_ORDER = ['200万円未満','200～250万円','250～300万円','300～350万円','350～400万円',
                '400～450万円','450～500万円','500～550万円','550～600万円','600～650万円',
                '650～700万円','700～750万円','750～800万円','800～900万円','900～1000万円',
                '1000～1250万円','1250～1500万円','1500万円以上']

pivot_savings = df_savings.pivot_table(index="年間収入階級", columns="項目", values="値", aggfunc="mean")
pivot_savings = pivot_savings.reindex(INCOME_ORDER)
pivot_savings["貯蓄/年収倍率"] = (pivot_savings["貯蓄"] / pivot_savings["年間収入"]).round(2)
pivot_savings["負債/年収倍率"] = (pivot_savings["負債"] / pivot_savings["年間収入"]).round(2)
pivot_savings.round(0)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 5))
x = range(len(INCOME_ORDER))
width = 0.4
ax.bar([i - width/2 for i in x], pivot_savings["貯蓄"], width=width, label="貯蓄")
ax.bar([i + width/2 for i in x], pivot_savings["負債"], width=width, label="負債")
ax.set_xticks(list(x))
ax.set_xticklabels(INCOME_ORDER, rotation=60, ha="right")
ax.set_ylabel("万円")
ax.set_title("年間収入階級別 貯蓄・負債(全期間平均)")
ax.legend()
plt.tight_layout()
plt.show()

## 4. 年収五分位別 黒字率の推移(2000〜2025年)

In [ ]:
df_surplus["年"] = df_surplus["時期"].astype(str).str[:4].astype(int)
QUINTILES = ["年収五分位1", "年収五分位2", "年収五分位3", "年収五分位4", "年収五分位5"]

rate_by_year = (
    df_surplus[df_surplus["項目"] == "黒字率"]
    .groupby(["年", "年間収入五分位"])["値"].mean()
    .unstack()[QUINTILES]
)

fig, ax = plt.subplots(figsize=(11, 5))
rate_by_year.plot(ax=ax)
ax.axvline(2020, color="gray", linestyle="--", linewidth=1, label="コロナ禍(2020)")
ax.set_ylabel("黒字率(%)")
ax.set_title("年収五分位別 黒字率の推移(勤労者世帯)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

print("直近5年の平均黒字率:")
rate_by_year.tail(5)

## 5. 可処分所得の使い道(消費 vs 貯蓄)

In [ ]:
summary = (
    df_surplus[df_surplus["項目"].isin(["平均消費性向", "黒字率"])]
    .groupby(["年間収入五分位", "項目"])["値"].mean()
    .unstack()
    .reindex(QUINTILES)
)
print(summary.round(1))

fig, ax = plt.subplots(figsize=(9, 5))
bottom = summary["平均消費性向"]
ax.bar(summary.index, summary["平均消費性向"], label="消費に回る割合(平均消費性向)")
ax.bar(summary.index, summary["黒字率"], bottom=bottom, label="貯蓄に回る割合(黒字率)")
ax.set_ylabel("%")
ax.set_title("年収五分位別 可処分所得の使い道")
ax.legend()
plt.tight_layout()
plt.show()

## 6. 世帯主の年齢階級別 貯蓄・負債(直近年)

同じ貯蓄・負債データを年齢の軸で見る。若年層は住宅ローンで純貯蓄(貯蓄−負債)が大きくマイナス、
50代でプラスに転じ、60代で貯蓄がピークになるライフサイクルが読み取れる。

In [ ]:
AGE_ORDER = ["29歳以下", "30～39歳", "40～49歳", "50～59歳", "60～69歳", "70歳以上"]

df_savings_age["年"] = df_savings_age["時期"].astype(str).str[:4].astype(int)
LATEST_YEAR = df_savings_age["年"].max()

pivot_age = (
    df_savings_age[df_savings_age["年"] == LATEST_YEAR]
    .pivot_table(index="年齢階級", columns="項目", values="値", aggfunc="mean")
    .reindex(AGE_ORDER)
)
pivot_age["純貯蓄"] = pivot_age["貯蓄"] - pivot_age["負債"]
print(f"{LATEST_YEAR}年平均(万円)")
print(pivot_age.round(0))

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(AGE_ORDER))
width = 0.4
ax.bar([i - width/2 for i in x], pivot_age["貯蓄"], width=width, label="貯蓄")
ax.bar([i + width/2 for i in x], pivot_age["負債"], width=width, label="負債")
ax.plot(list(x), pivot_age["純貯蓄"], color="black", marker="o", label="純貯蓄(貯蓄-負債)")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(AGE_ORDER)
ax.set_ylabel("万円")
ax.set_title(f"世帯主の年齢階級別 貯蓄・負債({LATEST_YEAR}年平均)")
ax.legend()
plt.tight_layout()
plt.show()

## 7. 同世代との比較

自分の年齢と貯蓄額を入れると、同世代(世帯主の年齢階級が同じ二人以上の世帯)の直近年平均と比較する。

> 注意: 公表されているのは**平均値**のみ。平均は一部の高資産世帯に引っ張られて高く出るため
> (家計調査では中央値は平均の6〜7割程度)、「平均より少ない=同世代の半分より下」ではない。

In [ ]:
MY_AGE = 35        # ← 自分の年齢に書き換える
MY_SAVINGS = 800   # ← 自分の貯蓄額(万円)に書き換える


def find_age_class10(age: int) -> str:
    """貯蓄・負債データ(10歳刻み)の階級名を返す"""
    if age <= 29:
        return "29歳以下"
    if age >= 70:
        return "70歳以上"
    lower = age // 10 * 10
    return f"{lower}～{lower + 9}歳"


def find_age_class5(age: int) -> str:
    """月々の収支データの階級名を返す(34歳以下は2015年以降まとめて公表)"""
    if age <= 34:
        return "34歳以下"
    lower = age // 5 * 5
    return f"{lower}～{lower + 4}歳"


# --- 資産(ストック)の比較: 貯蓄・負債データ(10歳刻み) ---
my_class = find_age_class10(MY_AGE)
peer = pivot_age.loc[my_class]
diff = MY_SAVINGS - peer["貯蓄"]

print(f"■ 資産の比較({LATEST_YEAR}年平均, 二人以上の世帯)")
print(f"  あなたの年齢階級  : {my_class}")
print(f"  同世代の平均貯蓄  : {peer['貯蓄']:,.0f}万円")
print(f"  同世代の平均負債  : {peer['負債']:,.0f}万円(大半は住宅ローン)")
print(f"  同世代の平均純貯蓄: {peer['純貯蓄']:,.0f}万円")
print(f"  あなたの貯蓄      : {MY_SAVINGS:,.0f}万円")
print(f"  平均との差        : {diff:+,.0f}万円(平均の{MY_SAVINGS / peer['貯蓄']:.2f}倍)")

# --- 毎月のペース(フロー)の比較: 月々の収支データ(直近12か月) ---
my_class5 = find_age_class5(MY_AGE)
recent_periods = sorted(df_flow_age["時期"].unique())[-12:]
flow_recent = (
    df_flow_age[df_flow_age["時期"].isin(recent_periods)]
    .pivot_table(index="年齢階級", columns="項目", values="値", aggfunc="mean")
)

print(f"\n■ 毎月のペースの比較(直近12か月平均, 勤労者世帯)")
if my_class5 in flow_recent.index:
    fp = flow_recent.loc[my_class5]
    print(f"  あなたの年齢階級      : {my_class5}")
    print(f"  同世代の平均可処分所得: {fp['可処分所得']:,.0f}円/月")
    print(f"  同世代の平均黒字      : {fp['黒字']:,.0f}円/月(年換算 約{fp['黒字'] * 12 / 10000:,.0f}万円)")
    print(f"  同世代の平均黒字率    : {fp['黒字率']:.1f}%")
else:
    print(f"  {my_class5} の勤労者世帯データは未公表(サンプル不足)")

## 8. 年齢階級別 貯蓄の内訳(何で持っているか)

貯蓄の中身を「通貨性預貯金(普通預金など)」「定期性預貯金」「生命保険など」「有価証券(株・投信・債券)」
「金融機関外(社内預金など)」の5つに分解する。年齢が上がるほど総額が増えるだけでなく、
**構成**がどう変わるか(有価証券の比率など)に注目。

In [ ]:
BREAKDOWN_ORDER = ["通貨性預貯金", "定期性預貯金", "生命保険など", "有価証券", "金融機関外"]

df_breakdown["年"] = df_breakdown["時期"].astype(str).str[:4].astype(int)
pivot_breakdown = (
    df_breakdown[df_breakdown["年"] == LATEST_YEAR]
    .pivot_table(index="年齢階級", columns="項目", values="値", aggfunc="mean")
    .reindex(index=AGE_ORDER, columns=BREAKDOWN_ORDER)
)

ratio = pivot_breakdown.div(pivot_breakdown.sum(axis=1), axis=0) * 100
print(f"{LATEST_YEAR}年平均の構成比(%)")
print(ratio.round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bottom = None
for col in BREAKDOWN_ORDER:
    axes[0].bar(pivot_breakdown.index, pivot_breakdown[col], bottom=bottom, label=col)
    bottom = pivot_breakdown[col] if bottom is None else bottom + pivot_breakdown[col]
axes[0].set_ylabel("万円")
axes[0].set_title(f"貯蓄の内訳・金額({LATEST_YEAR}年平均)")
axes[0].legend(fontsize=8)

bottom = None
for col in BREAKDOWN_ORDER:
    axes[1].bar(ratio.index, ratio[col], bottom=bottom, label=col)
    bottom = ratio[col] if bottom is None else bottom + ratio[col]
axes[1].set_ylabel("%")
axes[1].set_title("貯蓄の内訳・構成比")
axes[1].legend(fontsize=8)

for ax in axes:
    ax.set_xticklabels(AGE_ORDER, rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 9. 年齢別の毎月の貯蓄ペース(勤労者世帯)

「同世代は毎月いくら貯めているか」をフロー側から見る。
黒字=可処分所得−消費支出(貯蓄・投資・ローン返済などに回るお金)。

> - 34歳以下は2015年以降まとめて公表(それ以前は5歳刻み)。35歳以上は5歳刻み
> - 70歳以上の勤労者世帯はサンプルが少なく未公表のため、65〜69歳までの表示

In [ ]:
AGE5_ORDER = ["34歳以下", "35～39歳", "40～44歳", "45～49歳",
              "50～54歳", "55～59歳", "60～64歳", "65～69歳"]

flow_by_age = flow_recent.reindex(AGE5_ORDER)  # セクション7で作った直近12か月平均
print("直近12か月平均(円/月)")
print(flow_by_age[["可処分所得", "消費支出", "黒字", "黒字率"]].round(0))

fig, ax1 = plt.subplots(figsize=(10, 5))
x = range(len(AGE5_ORDER))
ax1.bar(x, flow_by_age["消費支出"], label="消費支出", color="tab:orange")
ax1.bar(x, flow_by_age["黒字"], bottom=flow_by_age["消費支出"], label="黒字(貯蓄などに回る)", color="tab:blue")
ax1.set_ylabel("円/月")
ax1.set_xticks(list(x))
ax1.set_xticklabels(AGE5_ORDER, rotation=30, ha="right")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(list(x), flow_by_age["黒字率"], color="black", marker="o", label="黒字率(右軸)")
ax2.set_ylabel("黒字率(%)")
ax2.legend(loc="upper right")

ax1.set_title("年齢別 可処分所得の使い道と黒字率(直近12か月平均)")
plt.tight_layout()
plt.show()